# Synthetic memory-graph contract verification

Fixed-seed fictional sources exercise local release/consumer gates. Recovery uses owned child processes that exit with os._exit, leaving real dead-owner locks. Rejections are checked against independent control bytes and publication-directory absence. No live source synchronization or model pipeline is included. The declared scope is not privacy telemetry.

Run from graph-memory/ or set GRAPH_MEMORY_MODULE_DIR to core/. Optional GRAPH_MEMORY_RESULT_FILE records a path-free result. Only newly allocated synthetic temporary sandboxes are changed and cleaned.


In [ ]:
import hashlib
import importlib
import json
import os
import random
import shutil
import stat
import subprocess
import sys
import tempfile
from pathlib import Path

sys.dont_write_bytecode = True
MODULE_DIR = Path(os.environ.get("GRAPH_MEMORY_MODULE_DIR", str(Path.cwd() / "core"))).resolve(strict=True)
if not MODULE_DIR.is_dir():
    raise RuntimeError("Explicit module directory is not a directory")
BRIEFING_DIR = MODULE_DIR.parent / "briefing"
for directory in (MODULE_DIR, BRIEFING_DIR):
    sys.path.insert(0, str(directory))
contract = importlib.import_module("memory_graph_release_contract_v4")
resolver = importlib.import_module("memory_graph_active_resolver_v4")
sealer = importlib.import_module("seal_memory_graph_release_v4")
activate = importlib.import_module("memory_graph_activate_v4")
briefing = importlib.import_module("memory_graph_briefing_v4")
SEED = 42
FIXED_TIME = "2000-01-01T00:00:00+00:00"

def check(condition, message):
    if not condition:
        raise RuntimeError(message)

def expect_failure(action, fragment):
    try:
        action()
    except Exception as exc:
        check(fragment in str(exc), "unexpected failure type or reason: " + str(exc))
        return
    raise RuntimeError("required failure was not raised: " + fragment)

def digest(raw):
    return hashlib.sha256(raw).hexdigest()

def json_bytes(value):
    return (json.dumps(value, ensure_ascii=False, indent=2) + "\n").encode("utf-8")

def write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(json_bytes(value))

def source_key(path):
    return os.path.normcase(os.path.normpath(str(path.resolve(strict=True))))

def evidence(epoch=7, health="PASS", registry="synthetic-registry", identity="synthetic-index"):
    return {
        "schema": "memory-graph-external-bti-evidence/v4",
        "mode": "STABLE_MUTABLE_REGISTRY",
        "registry_id": registry,
        "logical_identity": identity,
        "observed_applied_epoch": epoch,
        "health_status": health,
        "fingerprint": digest(("synthetic-seed-42-epoch-" + str(epoch)).encode()),
    }

class SyntheticRegistry:
    def __init__(self, value):
        self.value = dict(value)
        self.calls = 0

    def snapshot(self):
        self.calls += 1
        return dict(self.value)


In [ ]:
def synthetic_graph(root):
    labels = [
        "Current memory authority", "Source provenance policy", "Append preservation rule",
        "Release integrity contract", "Derived briefing navigation", "Transaction recovery policy",
    ]
    order = list(range(6))
    random.Random(SEED).shuffle(order)
    paths = [str(root / "a.md"), str(root / "b.md")]
    nodes = [
        {"id": "concept-" + str(i), "label": labels[i], "file_type": "document",
         "source_file": paths[i % 2], "source_location": "L2", "community": i % 3}
        for i in order
    ]
    pairs = [(0, 1, "requires"), (1, 2, "validates"), (2, 3, "preserves"),
             (3, 4, "authorizes"), (4, 5, "routes"), (5, 0, "recovers")]
    links = [
        {"source": "concept-" + str(a), "target": "concept-" + str(b),
         "relation": relation, "confidence": "EXTRACTED", "confidence_score": 0.95,
         "source_file": paths[index % 2], "source_location": "L2", "weight": 1.0}
        for index, (a, b, relation) in enumerate(pairs)
    ]
    groups = [(0, 1, 2), (2, 3, 4), (0, 4, 5)]
    hyperedges = [
        {"id": "route-" + str(index), "label": "Synthetic source-bound route " + str(index),
         "nodes": ["concept-" + str(i) for i in group], "relation": "coordinates",
         "confidence": "EXTRACTED", "confidence_score": 0.9,
         "source_file": paths[index % 2], "source_location": "L2"}
        for index, group in enumerate(groups)
    ]
    return {"nodes": nodes, "links": links, "hyperedges": hyperedges}

def create_root(parent, name):
    root = parent / name
    root.mkdir()
    (root / "a.md").write_bytes(b"# Synthetic policy\nThe current rule is verified against source evidence.\n")
    (root / "b.md").write_bytes(b"# Synthetic workflow\nDerived navigation must preserve append-only change signals.\n")
    (root / ".pending_update").write_bytes(b"# synthetic queue\na.md\nb.md\n")
    (root / ".rebuild_runs").mkdir()
    (root / ".graphify_generations").mkdir()
    return root

def make_candidate(root, name, prefix_count_delta=0, report_bytes=None):
    run_id = "run-" + name
    generation_dir = root / ".graphify_generations" / ("candidate-run-" + name)
    run_dir = root / ".rebuild_runs" / run_id
    canonical_dir = generation_dir / "canonical"
    graph_dir = generation_dir / "graphify-out"
    run_dir.mkdir()
    canonical_dir.mkdir(parents=True)
    graph_dir.mkdir()
    source_paths = [root / "a.md", root / "b.md"]
    corpus = [
        {"key": source_key(path), "path": str(path), "sha256": digest(path.read_bytes()), "bytes": len(path.read_bytes())}
        for path in source_paths
    ]
    pending_raw = (root / ".pending_update").read_bytes()
    prefix = {
        "exists": True, "path": str(root / ".pending_update"), "prefix_bytes": len(pending_raw),
        "prefix_sha256": digest(pending_raw),
        "non_comment_count": sum(1 for line in pending_raw.decode("utf-8").splitlines() if line.strip() and not line.strip().startswith("#")) + prefix_count_delta,
        "ends_with_newline": True,
    }
    plan = {
        "schema": "memory-graph-incremental-plan/v2", "memory_root": str(root), "run_id": run_id,
        "corpus": corpus, "corpus_manifest_sha256": contract.manifest_sha256(corpus),
        "pending_prefix": prefix,
        "safety": {"live_graph_activation": "disabled", "baseline_promotion": "disabled",
                   "pending_prefix_commit": "disabled", "legacy_flat_chunks": "read-only"},
    }
    plan_path = run_dir / "plan.json"
    write_json(plan_path, plan)
    graph = synthetic_graph(root)
    graph["nodes"][0]["label"] += " [" + name + "]"
    list_path = canonical_dir / ".graphify_chunk_0_files.txt"
    result_path = canonical_dir / ".graphify_chunk_0_result.json"
    list_path.write_bytes(("\n".join(str(path) for path in source_paths) + "\n").encode())
    write_json(result_path, {"nodes": graph["nodes"], "edges": graph["links"], "hyperedges": graph["hyperedges"]})
    chunks = [{
        "index": 0, "files_path": "canonical/.graphify_chunk_0_files.txt",
        "files_sha256": digest(list_path.read_bytes()), "result_path": "canonical/.graphify_chunk_0_result.json",
        "result_sha256": digest(result_path.read_bytes()), "sources": [item["key"] for item in corpus],
    }]
    generation = {
        "schema": "memory-graph-canonical-generation/v1", "state": "CANDIDATE", "created_at": FIXED_TIME,
        "run_id": run_id, "run_dir": str(run_dir), "memory_root": str(root),
        "plan_sha256": digest(plan_path.read_bytes()), "corpus": corpus,
        "corpus_manifest_sha256": contract.manifest_sha256(corpus), "pending_prefix": prefix,
        "canonical_chunks": chunks, "canonical_chunks_manifest_sha256": contract.manifest_sha256(chunks),
        "activation": "disabled", "baseline_promotion": "disabled", "pending_prefix_commit": "disabled",
    }
    generation_path = generation_dir / "generation_manifest.json"
    write_json(generation_path, generation)
    write_json(graph_dir / "graph.json", graph)
    (graph_dir / "GRAPH_REPORT.md").write_bytes(report_bytes if report_bytes is not None else ("# Synthetic release report\nGeneration " + name + ". Synthetic contract fixture only.\n").encode())
    (graph_dir / "graph.html").write_bytes(b"<!doctype html><html><body><p>Synthetic contract fixture. No renderer or network resource.</p></body></html>\n")
    outputs = [
        {"name": filename, "sha256": digest((graph_dir / filename).read_bytes()),
         "bytes": (graph_dir / filename).stat().st_size}
        for filename in ("graph.json", "GRAPH_REPORT.md", "graph.html")
    ]
    write_json(graph_dir / "graph_build_manifest.json", {
        "schema": "memory-graph-staged-build/v1", "state": "STAGED_NOT_ACTIVE",
        "created_at": FIXED_TIME, "generation_manifest_sha256": digest(generation_path.read_bytes()),
        "nodes": 6, "edges": 6, "communities": 3, "filter_audit": {},
        "outputs": outputs, "activation": "disabled",
    })
    write_json(run_dir / "run_state.json", {
        "schema": "memory-graph-incremental-run-state/v1", "state": "GRAPH_STAGED",
        "updated_at": FIXED_TIME, "run_id": run_id, "plan_sha256": digest(plan_path.read_bytes()),
        "candidate_generation": str(generation_dir), "staged_graph": str(graph_dir),
        "activation": "disabled", "baseline_promotion": "disabled", "pending_prefix_commit": "disabled",
        "generation_manifest_sha256": digest(generation_path.read_bytes()),
    })
    return {"path": generation_dir, "prefix": prefix, "corpus": corpus}

def make_fixture(parent, name, activate_now=True):
    root = create_root(parent, name)
    candidate = make_candidate(root, name)
    with (root / ".pending_update").open("ab") as handle:
        handle.write(b"late.md\n")
    pending_before = (root / ".pending_update").read_bytes()
    registry = SyntheticRegistry(evidence(epoch=8))
    sealed = sealer.seal_release(root, release_id="release-" + name, target_epoch=1,
                                 generation_dir=candidate["path"], bti_evidence=evidence())
    if activate_now:
        activated = activate.activate_release(root, Path(sealed["release_dir"]), expected_epoch=0, bti_adapter=registry)
        check(activated["status"] == "ACTIVE_VERIFIED", "wrong activation outcome")
    return {"root": root, "candidate": candidate, "sealed": sealed, "registry": registry, "pending_before": pending_before}

def output_bindings(directory):
    return {name: (directory / name).read_bytes() for name in
            ("quickstart.md", "briefing_state.json", "briefing_audit.json", "active_report_head_100.md")}

def cleanup_sandbox(path, temporary_parent):
    path = path.resolve(strict=True)
    check(path.parent == temporary_parent and path.name.startswith("dgm-"), "refused cleanup outside named sandbox")
    for child in path.rglob("*"):
        check(not child.is_symlink(), "unexpected sandbox link")
        if child.is_file():
            os.chmod(child, stat.S_IREAD | stat.S_IWRITE)
    shutil.rmtree(path)

def control_snapshot(root):
    names = [
        ".active_generation.json", ".active_generation.required.json", ".pending_update",
        ".memory_graph_activation_journal_v4.json", ".memory_graph_activation_v4.lock",
        ".incremental_rebuild_baseline.json", ".active_corpus_manifest_v4.json",
        ".legacy_projection_v4.json",
    ]
    snapshot = {name: (root / name).read_bytes() if (root / name).exists() else None for name in names}
    for folder in ("graphify-out", ".activation_journals_v4"):
        directory = root / folder
        snapshot[folder] = {p.relative_to(directory).as_posix(): p.read_bytes()
                            for p in directory.rglob("*") if p.is_file()} if directory.exists() else None
    snapshot["flat_chunks"] = {p.name: p.read_bytes() for p in root.glob(".graphify_chunk_*") if p.is_file()}
    return snapshot

def reject_unchanged(root, action, fragment):
    before = control_snapshot(root)
    caught = None
    try:
        action()
    except Exception as exc:
        caught = exc
    check(control_snapshot(root) == before, "rejected action changed independent control bytes")
    check(caught is not None, "required rejection did not occur: " + fragment)
    check(fragment in str(caught), "wrong rejection branch: " + str(caught))
    return caught

def reject_before_publish(root, release_id, action, fragment):
    target = root / ".graphify_releases" / release_id
    check(not target.exists(), "negative fixture already has a release")
    before = control_snapshot(root)
    caught = None
    try:
        action()
    except Exception as exc:
        caught = exc
    check(not target.exists(), "rejected candidate left a sealed release")
    check(control_snapshot(root) == before, "rejected candidate changed control bytes")
    check(caught is not None, "required candidate rejection did not occur")
    check(fragment in str(caught), "wrong candidate rejection branch: " + str(caught))

HARD_CRASH_CHILD = r"""
import json, os, sys
from pathlib import Path
sys.dont_write_bytecode = True
sys.path.insert(0, os.environ["GRAPH_MEMORY_MODULE_DIR"])
import memory_graph_activate_v4 as activate
request = json.loads(Path(os.environ["GRAPH_CHILD_REQUEST"]).read_text(encoding="utf-8"))
class Adapter:
    def snapshot(self):
        return dict(request["evidence"])
def hard_exit(name, expected):
    if name == expected:
        os._exit(9)
activate._failpoint = hard_exit
activate.activate_release(Path(request["root"]), Path(request["release_dir"]),
                          expected_epoch=0, bti_adapter=Adapter(), failpoint=request["failpoint"])
raise SystemExit("hard-exit checkpoint was not reached")
"""

def hard_crash_fixture(fixture, failpoint):
    request_path = fixture["root"] / "child-request.json"
    write_json(request_path, {"root": str(fixture["root"]), "release_dir": fixture["sealed"]["release_dir"],
                             "evidence": evidence(epoch=8), "failpoint": failpoint})
    child_env = dict(os.environ, GRAPH_MEMORY_MODULE_DIR=str(MODULE_DIR),
                     GRAPH_CHILD_REQUEST=str(request_path))
    args = [sys.executable, "-B"]
    if not __debug__:
        args.append("-O")
    args += ["-I", "-S", "-c", HARD_CRASH_CHILD]
    process = subprocess.Popen(args, env=child_env, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                               creationflags=subprocess.CREATE_NO_WINDOW if os.name == "nt" else 0)
    try:
        stdout, stderr = process.communicate(timeout=20)
    except subprocess.TimeoutExpired:
        process.kill()
        process.communicate()
        raise RuntimeError("owned hard-crash child timed out")
    check(process.returncode == 9, "owned child did not exit at the requested hard-crash checkpoint")
    lock_path = fixture["root"] / ".memory_graph_activation_v4.lock"
    journal_path = fixture["root"] / ".memory_graph_activation_journal_v4.json"
    check(lock_path.is_file() and journal_path.is_file(), "hard crash did not leave a real lock and journal")
    lock = json.loads(lock_path.read_bytes())
    check(lock["pid"] == process.pid, "leftover lock is not owned by the exited child")
    journal = json.loads(journal_path.read_bytes())
    return process.pid, journal

def verify_recovery_terminal(fixture, journal):
    root = fixture["root"]
    marker_path = root / ".active_generation.required.json"
    check(marker_path.is_file(), "recovery omitted the required marker")
    marker = json.loads(marker_path.read_bytes())
    check(marker == {"schema": "memory-graph-pointer-required/v4", "required": True}, "recovery omitted the required marker")
    projection_path = root / ".legacy_projection_v4.json"
    check(projection_path.is_file(), "recovery omitted the compatibility projection")
    projection = json.loads(projection_path.read_bytes())
    check(projection["release_id"] == fixture["sealed"]["release_id"] and projection["epoch"] == 1,
          "recovery omitted the intended compatibility projection")
    candidate_graph = fixture["candidate"]["path"] / "graphify-out"
    for name in ("graph.json", "GRAPH_REPORT.md", "graph.html"):
        check((root / "graphify-out" / name).read_bytes() == (candidate_graph / name).read_bytes(),
              "compatibility projection differs from frozen candidate bytes")
    candidate_canonical = fixture["candidate"]["path"] / "canonical"
    for path in candidate_canonical.iterdir():
        check((root / path.name).read_bytes() == path.read_bytes(), "flat canonical projection missing or wrong")
    check(not (root / ".memory_graph_activation_journal_v4.json").exists(), "recovery left the journal open")
    check(not (root / ".memory_graph_activation_v4.lock").exists(), "recovery left an activation lock")
    archived = root / ".activation_journals_v4" / (journal["tx_id"] + ".json")
    check(archived.is_file(), "recovery did not archive its own transaction")
    archived_value = json.loads(archived.read_bytes())
    check(archived_value["tx_id"] == journal["tx_id"] and archived_value["state"] == "COMMITTED",
          "recovery journal archive is not terminal")
    check((root / ".pending_update").read_bytes() == fixture["pending_before"], "recovery changed queue bytes")


In [ ]:
checks = []
hard_crash_exit_codes = []
dead_owner_probe_count = 0
temporary_parent = Path(tempfile.gettempdir()).resolve(strict=True)
sandbox = Path(tempfile.mkdtemp(prefix="dgm-", dir=str(temporary_parent)))
try:
    epoch_root = create_root(sandbox, "epoch-type")
    epoch_candidate = make_candidate(epoch_root, "epoch-type")
    for bad in [True, False, 1.0, "1", 0, -1]:
        reject_unchanged(epoch_root, lambda value=bad: sealer.seal_release(epoch_root, release_id="release-epoch-type", target_epoch=value, generation_dir=epoch_candidate["path"], bti_evidence=evidence()), "target epoch must be an integer >= 1")
    checks.append("seal_python_api_rejects_non_integer_and_non_positive_epoch")
    happy = make_fixture(sandbox, "happy")
    root = happy["root"]
    view = resolver.resolve_active_view(root)
    check(view["mode"] == "ACTIVE_V4" and view["epoch"] == 1, "wrong active view")
    check(view["pending_high_water"]["prefix_non_comment_count"] == 2, "wrong frozen prefix")
    check(view["pending_high_water"]["unconsumed_non_comment_count"] == 1, "late suffix lost")
    check(view["pending_high_water"]["prefix_bytes"] == happy["candidate"]["prefix"]["prefix_bytes"], "high-water raised")
    check((root / ".pending_update").read_bytes() == happy["pending_before"], "queue changed during activation")
    check(happy["registry"].calls == 1, "registry adapter was coupled to graph writes")
    checks.append("single_generation_seal_activate_exact_high_water_and_suffix")

    output = root / "derived"
    check(briefing.build_briefing(root, output)["status"] == "BUILT", "briefing not built")
    check(briefing.status_briefing(root, output)["status"] == "FRESH", "briefing status failed")
    text = briefing.consume_briefing(root, output)
    check(2000 <= len(text) <= 4000, "briefing shape changed")
    # Inputs in this fixture use English labels; source languages are not restricted.
    quickstart_cjk_count = sum(1 for character in text if (0x3400 <= ord(character) <= 0x4DBF) or (0x4E00 <= ord(character) <= 0x9FFF))
    check(quickstart_cjk_count == 0, "synthetic quickstart contains untranslated fixed display text")
    check(";  Nearby entries:" not in text and ";  First entries:" not in text, "double spaces remain in fixed display joins")
    check(".Non-AMBIGUOUS" not in text and "/3.Missing" not in text, "fixed display sentence space missing")
    quickstart_characters = len(text)
    before_cache = output_bindings(output)
    with (root / ".pending_update").open("ab") as handle:
        handle.write(b"later.md\n")
    check(resolver.resolve_active_view(root)["pending_high_water"]["unconsumed_non_comment_count"] == 2, "append not counted")
    check(briefing.build_briefing(root, output)["status"] == "NOOP", "valid append rewrote cache")
    check(output_bindings(output) == before_cache, "same-release cache bytes changed")
    check(briefing.consume_briefing(root, output) == text, "same-release consume changed")
    checks.append("briefing_build_status_consume_and_valid_append_noop")

    (output / "quickstart.md").write_bytes(before_cache["quickstart.md"] + b"altered\n")
    reject_unchanged(root, lambda: briefing.consume_briefing(root, output), "hash mismatch")
    checks.append("derived_cache_tamper_rejected")

    modified = make_fixture(sandbox, "prefix-modified")
    queue = modified["root"] / ".pending_update"
    raw = queue.read_bytes()
    queue.write_bytes(b"X" + raw[1:])
    reject_unchanged(modified["root"], lambda: resolver.resolve_active_view(modified["root"]), "prefix hash mismatch")
    checks.append("prefix_modification_rejected")

    truncated = make_fixture(sandbox, "prefix-truncated")
    (truncated["root"] / ".pending_update").write_bytes(b"x")
    reject_unchanged(truncated["root"], lambda: resolver.resolve_active_view(truncated["root"]), "shorter than frozen prefix")
    checks.append("prefix_truncation_rejected")

    missing = make_fixture(sandbox, "pointer-missing")
    check((missing["root"] / "graphify-out/graph.json").exists(), "legacy projection absent")
    (missing["root"] / ".active_generation.json").unlink()
    reject_unchanged(missing["root"], lambda: resolver.resolve_active_view(missing["root"], pointer_required=False), "pointer-required marker")
    checks.append("required_marker_blocks_missing_pointer_legacy_fallback")

    tampered = make_fixture(sandbox, "release-tampered")
    graph_file = Path(tampered["sealed"]["release_dir"]) / "graphify-out/graph.json"
    os.chmod(graph_file, stat.S_IREAD | stat.S_IWRITE)
    graph_file.write_bytes(graph_file.read_bytes() + b" ")
    reject_unchanged(tampered["root"], lambda: resolver.resolve_active_view(tampered["root"]), "hash mismatch")
    checks.append("sealed_artifact_tamper_rejected")

    # Both contenders are sealed for the same target epoch before a winner commits.
    cas_root = create_root(sandbox, "cas")
    contender_a = make_candidate(cas_root, "cas-first")
    contender_b = make_candidate(cas_root, "cas-second")
    release_a = sealer.seal_release(cas_root, release_id="release-cas-first", target_epoch=1, generation_dir=contender_a["path"], bti_evidence=evidence())
    release_b = sealer.seal_release(cas_root, release_id="release-cas-second", target_epoch=1, generation_dir=contender_b["path"], bti_evidence=evidence())
    activate.activate_release(cas_root, Path(release_a["release_dir"]), expected_epoch=0, bti_adapter=SyntheticRegistry(evidence(epoch=8)))
    winner_pointer = (cas_root / ".active_generation.json").read_bytes()
    reject_unchanged(cas_root, lambda: activate.activate_release(cas_root, Path(release_b["release_dir"]), expected_epoch=0, bti_adapter=SyntheticRegistry(evidence(epoch=8))), "expected epoch mismatch")
    check((cas_root / ".active_generation.json").read_bytes() == winner_pointer, "stale contender replaced winner")
    check(resolver.resolve_active_view(cas_root)["release_id"] == "release-cas-first", "stale contender changed active identity")
    checks.append("same_target_stale_cas_preserves_winner")

    for label, value, message in [
        ("health", evidence(health="FAIL"), "health is not PASS"),
        ("identity", evidence(identity="different-synthetic-index"), "logical_identity mismatch"),
        ("epoch", evidence(epoch=6), "epoch regressed"),
        ("registry", evidence(registry="different-synthetic-registry"), "registry_id mismatch"),
    ]:
        failed = make_fixture(sandbox, "registry-" + label, activate_now=False)
        reject_unchanged(failed["root"], lambda f=failed, v=value: activate.activate_release(f["root"], Path(f["sealed"]["release_dir"]), expected_epoch=0, bti_adapter=SyntheticRegistry(v)), message)
        check(not (failed["root"] / ".active_generation.json").exists(), "failed preflight committed a pointer")
    checks.append("registry_health_identity_id_and_epoch_fail_closed")

    # An in-process interruption leaves a journal but cleans its lock; this isolates the journal precondition.
    journal_only = make_fixture(sandbox, "journal-block", activate_now=False)
    expect_failure(lambda: activate.activate_release(journal_only["root"], Path(journal_only["sealed"]["release_dir"]), expected_epoch=0, bti_adapter=journal_only["registry"], failpoint="after_journal"), "injected crash")
    check((journal_only["root"] / ".memory_graph_activation_journal_v4.json").is_file(), "interruption left no journal")
    check(not (journal_only["root"] / ".memory_graph_activation_v4.lock").exists(), "journal precondition is masked by a lock")
    reject_unchanged(journal_only["root"], lambda: activate.activate_release(journal_only["root"], Path(journal_only["sealed"]["release_dir"]), expected_epoch=0, bti_adapter=journal_only["registry"]), "recover first")
    checks.append("pending_journal_blocks_fresh_activation_without_side_effects")

    for failpoint in ("after_journal", "after_pointer", "after_marker", "after_projection"):
        interrupted = make_fixture(sandbox, "hard-" + failpoint, activate_now=False)
        child_pid, pending_journal = hard_crash_fixture(interrupted, failpoint)
        hard_crash_exit_codes.append(9)
        probe_calls = []
        original_pid_alive = activate._pid_alive
        def observed_pid_alive(pid):
            value = original_pid_alive(pid)
            probe_calls.append((pid, value))
            return value
        activate._pid_alive = observed_pid_alive
        try:
            recovered = activate.recover_activation(interrupted["root"], bti_adapter=interrupted["registry"])
        finally:
            activate._pid_alive = original_pid_alive
        check((child_pid, False) in probe_calls, "recovery did not probe the exited lock owner")
        dead_owner_probe_count += 1
        check(recovered["status"] == "RECOVERED_ACTIVE_VERIFIED", "wrong recovery outcome")
        verify_recovery_terminal(interrupted, pending_journal)
        check(resolver.resolve_active_view(interrupted["root"])["release_id"] == interrupted["sealed"]["release_id"], "recovery selected wrong release")
    checks.append("hard_crash_dead_owner_and_all_four_recovery_terminal_states")

    live = make_fixture(sandbox, "live-owner-lock", activate_now=False)
    holder = subprocess.Popen([sys.executable, "-I", "-S", "-c", "import time; time.sleep(60)"],
                              creationflags=subprocess.CREATE_NO_WINDOW if os.name == "nt" else 0)
    try:
        (live["root"] / ".memory_graph_activation_v4.lock").write_bytes(json_bytes({"schema": activate.LOCK_SCHEMA, "pid": holder.pid, "nonce": "0" * 32, "created_at": FIXED_TIME}))
        reject_unchanged(live["root"], lambda: activate.recover_activation(live["root"], bti_adapter=live["registry"]), "owner is still alive")
        reject_unchanged(live["root"], lambda: activate.activate_release(live["root"], Path(live["sealed"]["release_dir"]), expected_epoch=0, bti_adapter=live["registry"]), "activation lock exists")
    finally:
        holder.kill()
        holder.communicate()
    checks.append("live_owner_lock_blocks_recovery_and_fresh_activation")

    mixed_root = create_root(sandbox, "mixed")
    first = make_candidate(mixed_root, "mixed-first")
    other = make_candidate(mixed_root, "mixed-other")
    check((first["path"] / "graphify-out/graph.json").read_bytes() != (other["path"] / "graphify-out/graph.json").read_bytes(), "two graph generations are identical")
    check((first["path"] / "graphify-out/GRAPH_REPORT.md").read_bytes() != (other["path"] / "graphify-out/GRAPH_REPORT.md").read_bytes(), "two report generations are identical")
    for name in ("graph.json", "GRAPH_REPORT.md", "graph.html", "graph_build_manifest.json"):
        shutil.copyfile(other["path"] / "graphify-out" / name, first["path"] / "graphify-out" / name)
    reject_before_publish(mixed_root, "release-mixed", lambda: sealer.seal_release(mixed_root, release_id="release-mixed", target_epoch=1, generation_dir=first["path"], bti_evidence=evidence()), "generation hash mismatch")
    checks.append("mixed_generation_rejected_without_published_directory")

    for filename, label in (("graph.json", "graph"), ("GRAPH_REPORT.md", "report")):
        mix = create_root(sandbox, "mix-" + label)
        one = make_candidate(mix, "mix-" + label + "-first")
        two = make_candidate(mix, "mix-" + label + "-second")
        a, b = one["path"] / "graphify-out" / filename, two["path"] / "graphify-out" / filename
        check(a.read_bytes() != b.read_bytes(), "artifact mix fixture is not distinct")
        shutil.copyfile(b, a)
        release_id = "release-mix-" + label
        reject_before_publish(mix, release_id, lambda r=mix, g=one, rid=release_id: sealer.seal_release(r, release_id=rid, target_epoch=1, generation_dir=g["path"], bti_evidence=evidence()), filename + " hash mismatch")
    checks.append("distinct_graph_and_report_swaps_rejected_before_publish")

    rebound = make_fixture(sandbox, "rebound")
    rebound_output = rebound["root"] / "derived"
    check(briefing.build_briefing(rebound["root"], rebound_output)["status"] == "BUILT", "old briefing not built")
    cache_before_rebind = output_bindings(rebound_output)
    old_report = (rebound["candidate"]["path"] / "graphify-out/GRAPH_REPORT.md").read_bytes()
    next_candidate = make_candidate(rebound["root"], "rebound-next", report_bytes=old_report)
    next_release = sealer.seal_release(rebound["root"], release_id="release-rebound-next", target_epoch=2, generation_dir=next_candidate["path"], bti_evidence=evidence())
    activate.activate_release(rebound["root"], Path(next_release["release_dir"]), expected_epoch=1, bti_adapter=SyntheticRegistry(evidence(epoch=8)))
    check(resolver.resolve_active_view(rebound["root"])["epoch"] == 2, "new release did not activate")
    reject_unchanged(rebound["root"], lambda: briefing.consume_briefing(rebound["root"], rebound_output), "authority binding mismatch")
    reject_unchanged(rebound["root"], lambda: briefing.status_briefing(rebound["root"], rebound_output), "authority binding mismatch")
    check(output_bindings(rebound_output) == cache_before_rebind, "stale cache rejection rewrote cached evidence")
    checks.append("new_release_rejects_old_briefing_consume_and_status")

    corrupt = make_fixture(sandbox, "pointer-corrupt")
    check((corrupt["root"] / "graphify-out/graph.json").exists(), "legacy fallback fixture is absent")
    (corrupt["root"] / ".active_generation.json").write_bytes(b"{")
    reject_unchanged(corrupt["root"], lambda: resolver.resolve_active_view(corrupt["root"], pointer_required=False), "active pointer: invalid UTF-8 JSON")
    unmarked = make_fixture(sandbox, "pointer-corrupt-unmarked")
    (unmarked["root"] / ".active_generation.json").write_bytes(b"{")
    (unmarked["root"] / ".active_generation.required.json").unlink()
    check((unmarked["root"] / "graphify-out/graph.json").exists(), "legacy fallback fixture is absent")
    reject_unchanged(unmarked["root"], lambda: resolver.resolve_active_view(unmarked["root"], pointer_required=False), "active pointer: invalid UTF-8 JSON")
    checks.append("corrupt_pointer_never_falls_back")

    for label, marker_value, message in [
        ("false", {"schema": "memory-graph-pointer-required/v4", "required": False}, "invalid"),
        ("extra", {"schema": "memory-graph-pointer-required/v4", "required": True, "extra": 1}, "fields mismatch"),
    ]:
        broken_marker = make_fixture(sandbox, "marker-" + label)
        write_json(broken_marker["root"] / ".active_generation.required.json", marker_value)
        reject_unchanged(broken_marker["root"], lambda f=broken_marker: resolver.resolve_active_view(f["root"], pointer_required=False), message)
    checks.append("malformed_required_marker_rejected")

    for label, queue_bytes, count_delta, message in [
        ("lf", b"# synthetic queue\na.md\nb.md", 0, "not LF-delimited"),
        ("count", b"# synthetic queue\na.md\nb.md\n", 1, "line count mismatch"),
    ]:
        bad = create_root(sandbox, "high-water-" + label)
        (bad / ".pending_update").write_bytes(queue_bytes)
        candidate = make_candidate(bad, "high-water-" + label, prefix_count_delta=count_delta)
        release_id = "release-high-water-" + label
        reject_before_publish(bad, release_id, lambda r=bad, g=candidate, rid=release_id: sealer.seal_release(r, release_id=rid, target_epoch=1, generation_dir=g["path"], bti_evidence=evidence()), message)
    checks.append("high_water_lf_and_effective_line_count_rejected_before_publish")

    encoded = make_fixture(sandbox, "suffix-encoding")
    with (encoded["root"] / ".pending_update").open("ab") as handle:
        handle.write(b"\xff\n")
    reject_unchanged(encoded["root"], lambda: resolver.resolve_active_view(encoded["root"]), "pending suffix: invalid UTF-8")
    consumed = make_fixture(sandbox, "consume-suffix")
    consumed_output = consumed["root"] / "derived"
    check(briefing.build_briefing(consumed["root"], consumed_output)["status"] == "BUILT", "suffix briefing not built")
    with (consumed["root"] / ".pending_update").open("ab") as handle:
        handle.write(b"\xff\n")
    reject_unchanged(consumed["root"], lambda: briefing.consume_briefing(consumed["root"], consumed_output), "pending suffix: invalid UTF-8")
    checks.append("invalid_utf8_live_suffix_rejected")

    counted = make_fixture(sandbox, "suffix-count")
    with (counted["root"] / ".pending_update").open("ab") as handle:
        handle.write(b"# note\n\n   \nnext.md\n")
    check(resolver.resolve_active_view(counted["root"])["pending_high_water"]["unconsumed_non_comment_count"] == 2, "suffix comment or blank lines were counted")
    checks.append("suffix_count_excludes_comment_and_blank_lines")

    manifest_bound = make_fixture(sandbox, "manifest-bound")
    manifest_path = Path(manifest_bound["sealed"]["release_dir"]) / "release_manifest.json"
    value = json.loads(manifest_path.read_bytes())
    value["external_checks"]["bti"] = evidence(epoch=6)
    os.chmod(manifest_path, stat.S_IREAD | stat.S_IWRITE)
    write_json(manifest_path, value)
    os.chmod(manifest_path, stat.S_IREAD)
    reject_unchanged(manifest_bound["root"], lambda: resolver.resolve_active_view(manifest_bound["root"]), "release manifest hash mismatch")
    checks.append("pointer_rejects_rebound_release_manifest")

    conflict = make_fixture(sandbox, "recover-conflict")
    waiting_candidate = make_candidate(conflict["root"], "recover-waiting")
    winning_candidate = make_candidate(conflict["root"], "recover-other")
    waiting_release = sealer.seal_release(conflict["root"], release_id="release-recover-waiting", target_epoch=2, generation_dir=waiting_candidate["path"], bti_evidence=evidence())
    winning_release = sealer.seal_release(conflict["root"], release_id="release-recover-other", target_epoch=2, generation_dir=winning_candidate["path"], bti_evidence=evidence())
    expect_failure(lambda: activate.activate_release(conflict["root"], Path(waiting_release["release_dir"]), expected_epoch=1, bti_adapter=conflict["registry"], failpoint="after_journal"), "injected crash")
    journal_path = conflict["root"] / ".memory_graph_activation_journal_v4.json"
    waiting_journal = journal_path.read_bytes()
    journal_path.unlink()
    # Simulate an independently committed pointer, then restore the older interrupted transaction.
    activate.activate_release(conflict["root"], Path(winning_release["release_dir"]), expected_epoch=1, bti_adapter=conflict["registry"])
    journal_path.write_bytes(waiting_journal)
    reject_unchanged(conflict["root"], lambda: activate.recover_activation(conflict["root"], bti_adapter=conflict["registry"]), "pointer belongs to another transaction")
    check(resolver.resolve_active_view(conflict["root"])["release_id"] == "release-recover-other", "conflicting recovery replaced independent winner")
    checks.append("recovery_cas_conflict_preserves_independent_commit")

    check(contract.validate_bti_evidence(evidence(epoch=0))["observed_applied_epoch"] == 0, "integer zero evidence rejected")
    for bad_epoch in (True, False, 7.0, "7"):
        expect_failure(lambda value=bad_epoch: contract.validate_bti_evidence(evidence(epoch=value)), "observed_applied_epoch invalid")
    checks.append("external_evidence_epoch_is_json_integer_not_boolean")

finally:
    cleanup_sandbox(sandbox, temporary_parent)
check(not sandbox.exists(), "sandbox cleanup left a directory")

RESULT = {
    "schema": "dual-agent-memory-graph-synthetic-result/v1", "status": "PASS",
    "seed": SEED, "checks": checks, "check_groups": len(checks),
    "optimized_mode": not __debug__, "synthetic_quickstart_cjk_count": quickstart_cjk_count,
    "synthetic_quickstart_characters": quickstart_characters, "fixed_display_language": "English",
    "hard_crash_exit_codes": hard_crash_exit_codes, "dead_owner_probe_count": dead_owner_probe_count,
    "sandbox_cleaned": True,
    "declared_scope": {"inputs": "synthetic_sandbox_only", "network": "not_used_by_this_code",
                       "graphify_or_llm": "not_invoked", "registry": "in_memory_stub_only", "measured": False},
}
print(json.dumps(RESULT, sort_keys=True))
if os.environ.get("GRAPH_MEMORY_RESULT_FILE"):
    target = Path(os.environ["GRAPH_MEMORY_RESULT_FILE"])
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(json_bytes(RESULT))
